Import library

In [1]:
import spacy
from spacy import displacy

Load pretrained dependency parser

In [2]:
nlp = spacy.load("en_core_web_sm")

Parsing Algorithm with detailed output

In [9]:
def simulate_transition_parse(sentence):

    doc = nlp(sentence)

    words = [token.text for token in doc]
    heads = {token.i: token.head.i for token in doc}

    stack = ["ROOT"]
    buffer = list(range(len(words)))

    dependencies = []

    step = 1

    print(f"{'Step':<5} {'Stack':<40} {'Buffer':<50} {'Transition':<12} {'New Dependency'}")
    print("-"*140)

    while buffer or len(stack) > 1:

        stack_words = ["ROOT" if s=="ROOT" else words[s] for s in stack]
        buffer_words = [words[i] for i in buffer]

        transition = ""
        new_dep = ""

        if len(stack) >= 2:
            s1 = stack[-1]
            s2 = stack[-2]

            if s2 != "ROOT" and heads[s2] == s1:
                transition = "LEFT-ARC"
                new_dep = f"{words[s1]} → {words[s2]}"
                dependencies.append((words[s1], words[s2]))
                stack.pop(-2)

            elif s1 != "ROOT" and heads[s1] == s2:
                transition = "RIGHT-ARC"
                new_dep = f"{words[s2]} → {words[s1]}"
                dependencies.append((words[s2], words[s1]))
                stack.pop()

            else:
                if buffer:
                    transition = "SHIFT"
                    stack.append(buffer.pop(0))
                else:
                    stack.pop()

        else:
            transition = "SHIFT"
            stack.append(buffer.pop(0))

        print(f"{step:<5} {str(stack_words):<40} {str(buffer_words):<50} {transition:<12} {new_dep}")

        step += 1

    if stack[-1] != "ROOT":
        dependencies.append(("ROOT", words[stack[-1]]))

    print("\nFinal Dependencies:")
    for head, dep in dependencies:
        print(f"{head} → {dep}")

    return dependencies

Running the algo with a sentence

In [10]:
sentence = input("Enter a sentence: ")
simulate_transition_parse(sentence)
doc = nlp(sentence)
displacy.render(doc, style="dep", jupyter=True)

Step  Stack                                    Buffer                                             Transition   New Dependency
--------------------------------------------------------------------------------------------------------------------------------------------
1     ['ROOT']                                 ['I', 'parsed', 'this', 'sentence', 'correctly']   SHIFT        
2     ['ROOT', 'I']                            ['parsed', 'this', 'sentence', 'correctly']        SHIFT        
3     ['ROOT', 'I', 'parsed']                  ['this', 'sentence', 'correctly']                  LEFT-ARC     parsed → I
4     ['ROOT', 'parsed']                       ['this', 'sentence', 'correctly']                  SHIFT        
5     ['ROOT', 'parsed', 'this']               ['sentence', 'correctly']                          SHIFT        
6     ['ROOT', 'parsed', 'this', 'sentence']   ['correctly']                                      LEFT-ARC     sentence → this
7     ['ROOT', 'parsed', 'sentence']